In [1]:
import pandas as pd
df=pd.read_csv("100_Unique_QA_Dataset.csv")
print(df.head())

                                          question      answer
0                   What is the capital of France?       Paris
1                  What is the capital of Germany?      Berlin
2               Who wrote 'To Kill a Mockingbird'?  Harper-Lee
3  What is the largest planet in our solar system?     Jupiter
4   What is the boiling point of water in Celsius?         100


In [3]:
# Tokenize
def tokenize(text):
    return text.lower().replace("'", "").replace("?", "").split()

In [7]:
# vocab
vocab = {'<UNK>':0}
def build_vocab(df):
    tokenized_questions = df['question'].apply(tokenize)
    tokenized_answers = df['answer'].apply(tokenize)
    merged_tokens = tokenized_questions + tokenized_answers
    for tokens in merged_tokens:
        for word in tokens:
            if word not in vocab:
                vocab[word] = len(vocab)
    return vocab

In [8]:
vocab = build_vocab(df)

In [9]:
vocab

{'<UNK>': 0,
 'what': 1,
 'is': 2,
 'the': 3,
 'capital': 4,
 'of': 5,
 'france': 6,
 'paris': 7,
 'germany': 8,
 'berlin': 9,
 'who': 10,
 'wrote': 11,
 'to': 12,
 'kill': 13,
 'a': 14,
 'mockingbird': 15,
 'harper-lee': 16,
 'largest': 17,
 'planet': 18,
 'in': 19,
 'our': 20,
 'solar': 21,
 'system': 22,
 'jupiter': 23,
 'boiling': 24,
 'point': 25,
 'water': 26,
 'celsius': 27,
 '100': 28,
 'painted': 29,
 'mona': 30,
 'lisa': 31,
 'leonardo-da-vinci': 32,
 'square': 33,
 'root': 34,
 '64': 35,
 '8': 36,
 'chemical': 37,
 'symbol': 38,
 'for': 39,
 'gold': 40,
 'au': 41,
 'which': 42,
 'year': 43,
 'did': 44,
 'world': 45,
 'war': 46,
 'ii': 47,
 'end': 48,
 '1945': 49,
 'longest': 50,
 'river': 51,
 'nile': 52,
 'japan': 53,
 'tokyo': 54,
 'developed': 55,
 'theory': 56,
 'relativity': 57,
 'albert-einstein': 58,
 'freezing': 59,
 'fahrenheit': 60,
 '32': 61,
 'known': 62,
 'as': 63,
 'red': 64,
 'mars': 65,
 'author': 66,
 '1984': 67,
 'george-orwell': 68,
 'currency': 69,
 'unit

In [10]:
# Numerical Indices
def text_to_indices(text, vocab):
    indexed_text = []
    for token in tokenize(text):
        if token in vocab:
            indexed_text.append(vocab[token])
        else:
            indexed_text.append(vocab['<UNK>'])

    return indexed_text

In [11]:
import torch
from torch.utils.data import Dataset, DataLoader


In [12]:
class QADataset(Dataset):
    def __init__(self, df, vocab):
        self.df =df
        self.vocab = vocab
    
    def __len__(self):
        return self.df.shape[0]
    
    def __getitem__(self, idx):
        question = text_to_indices(self.df.iloc[idx]['question'], self.vocab)
        answer = text_to_indices(self.df.iloc[idx]['answer'], self.vocab)
        
        numerical_question_indices = question
        numerical_answer_indices = answer
        
        return torch.tensor(numerical_question_indices), torch.tensor(numerical_answer_indices)

In [14]:
dataset=QADataset(df, vocab)

In [15]:
dataloader= DataLoader(dataset, batch_size=1, shuffle=True)

In [16]:
from torch import nn
class SimpleRNN(nn.Module):
    def __init__(self, vocab_size):
        super(SimpleRNN, self).__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
        self.rnn = nn.RNN(50, 64, batch_first=True)
        self.fc = nn.Linear(64, vocab_size)
    
    
    def forward(self, question ):
        embedded_question = self.embedding(question)
        _, final = self.rnn(embedded_question)
        output = self.fc(final.squeeze(0))
        return output
    
    

In [17]:



learning_rate=0.001
epochs =20

In [18]:
model = SimpleRNN(vocab_size=len(vocab))

In [19]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [21]:
# training loop

for epoch in range(epochs):
    model.train()
    total_loss = 0
    for question, answer in dataloader:
        
        optimizer.zero_grad()
        output = model(question)
        loss = criterion(output, answer[0 ])
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
    
    print(f'Epoch {epoch+1}/{epochs}, Loss: {total_loss/len(dataloader):.4f}')


Epoch 1/20, Loss: 5.0688
Epoch 2/20, Loss: 4.2469
Epoch 3/20, Loss: 3.5312
Epoch 4/20, Loss: 2.9206
Epoch 5/20, Loss: 2.3726
Epoch 6/20, Loss: 1.8751
Epoch 7/20, Loss: 1.4469
Epoch 8/20, Loss: 1.1030
Epoch 9/20, Loss: 0.8444
Epoch 10/20, Loss: 0.6473
Epoch 11/20, Loss: 0.5036
Epoch 12/20, Loss: 0.3998
Epoch 13/20, Loss: 0.3249
Epoch 14/20, Loss: 0.2675
Epoch 15/20, Loss: 0.2212
Epoch 16/20, Loss: 0.1875
Epoch 17/20, Loss: 0.1605
Epoch 18/20, Loss: 0.1383
Epoch 19/20, Loss: 0.1226
Epoch 20/20, Loss: 0.1043


In [29]:
# Predict
def predict(model,question,threshold=0.5):
    
    numerical_question = text_to_indices(question,vocab)
    question_tensor = torch.tensor(numerical_question, dtype=torch.long).unsqueeze(0)
    output=model(question_tensor)
    # logits to probs
    probs = torch.nn.functional.softmax(output,dim=1)
    value,index = torch.max(probs,dim=1)
    if value<threshold:
        return "I don't know"
    print(list(vocab.keys())[index])

In [31]:
predict(model,"what is capital of france")

paris
